# `weather_data.parquet` processing example

This notebook works through **`team-members/dylan-han/weather_data.parquet`**: **fast loading**, **missing and outlier cleaning**, **combining multiple stations or months**, and **grouped analysis by station / year / month**.

**Fields:** `source_file` looks like `IDCJDW2012.202602.csv` and can be parsed into a station id and `YYYYMM`; `Date` is a string date column.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path("..") / "weather_data.parquet"
assert DATA_PATH.is_file(), f"Data file not found: {DATA_PATH.resolve()}"

## 1. Fast read

`read_parquet(..., engine="pyarrow")` is usually fastest when PyArrow is installed. Restricting columns with `columns=[...]` speeds things up further when you only need a subset.

In [ ]:
%%time
df_raw = pd.read_parquet(DATA_PATH, engine="pyarrow")
print("shape:", df_raw.shape)
display(df_raw.head(3))
df_raw.dtypes

## 2. Missing values and outliers

- Drop `Unnamed: 0` when it is entirely non-informative (all missing).
- Parse `station` and `file_year_month` from `source_file`; parse `Date` to `datetime` and derive `year` / `month`.
- Numeric columns: treat values outside **1.5×IQR** as outliers and set them to `NaN` (you can switch to winsorizing or row drops per project needs).
- Text columns: strip leading/trailing whitespace; coerce wind speed etc. to numeric where appropriate.

In [ ]:
def parse_source_file(s: str):
    """IDCJDW2012.202602.csv -> ('2012', 202602.0)"""
    if not isinstance(s, str) or not s:
        return (np.nan, np.nan)
    base = Path(s).stem  # IDCJDW2012.202602
    parts = base.split(".")
    if len(parts) != 2:
        return (np.nan, np.nan)
    head, yyyymm = parts
    if not head.upper().startswith("IDCJDW"):
        return (np.nan, np.nan)
    station = head[len("IDCJDW") :]
    try:
        yyyymm_int = int(yyyymm)
    except ValueError:
        yyyymm_int = np.nan
    return (station, float(yyyymm_int))


def iqr_mask_outliers(x: pd.Series, k: float = 1.5) -> pd.Series:
    """True = keep value; False = outlier (outside IQR fences)."""
    if x.dtype == object:
        return pd.Series(True, index=x.index)
    valid = x.dropna()
    if valid.empty:
        return pd.Series(True, index=x.index)
    q1, q3 = valid.quantile(0.25), valid.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return (x >= low) & (x <= high) | x.isna()


df = df_raw.copy()
if "Unnamed: 0" in df.columns and df["Unnamed: 0"].isna().all():
    df = df.drop(columns=["Unnamed: 0"])

parsed = df["source_file"].map(parse_source_file)
df["station"] = [p[0] for p in parsed]
df["file_year_month"] = [p[1] for p in parsed]

df["date"] = pd.to_datetime(df["Date"], format="mixed", errors="coerce")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

str_cols = df.select_dtypes(include=["object", "string"]).columns
for c in str_cols:
    df[c] = df[c].astype("string").str.strip()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in ["file_year_month", "year", "month"]:
    if c in numeric_cols:
        numeric_cols.remove(c)

outlier_counts = {}
for c in numeric_cols:
    mask = iqr_mask_outliers(df[c], k=1.5)
    n_bad = int((~mask).sum())
    if n_bad:
        outlier_counts[c] = n_bad
        df.loc[~mask, c] = np.nan

print("Missing counts per column (top 12 after cleaning):")
print(df.isna().sum().sort_values(ascending=False).head(12))
print("\nOutliers set to NaN by IQR (counts per column):", outlier_counts if outlier_counts else "none")

## 3. Combining multiple stations or months

This Parquet file is already a **row-wise stack** of many stations and months. The cells below show:

1. Filter a few `station` values and `concat` (equivalent to subsetting the full table).
2. For one `station`, sort by `date` and treat rows from multiple monthly files as one series (`drop_duplicates` on `date` to handle overlapping files).

In [ ]:
stations_sample = df["station"].dropna().unique()[:3]
parts = [df[df["station"] == st].copy() for st in stations_sample]
multi_station = pd.concat(parts, axis=0, ignore_index=True)
print("Sample stations:", list(stations_sample), "-> combined rows:", len(multi_station))

st = df["station"].dropna().mode().iloc[0]
one_station_ts = (
    df[df["station"] == st]
    .sort_values("date")
    .drop_duplicates(subset=["date"], keep="last")
    .reset_index(drop=True)
)
print(f"Station {st}: unique days after dedupe:", one_station_ts.shape[0])

## 4. Grouped analysis by station / year / month

Aggregate key numeric columns with group means and counts; swap in `median`, `min`, `max`, etc. as needed.

In [ ]:
value_cols = [
    c
    for c in [
        "Minimum temperature (°C)",
        "Maximum temperature (°C)",
        "Rainfall (mm)",
        "9am Temperature (°C)",
        "3pm Temperature (°C)",
    ]
    if c in df.columns
]

g = df.dropna(subset=["station", "year", "month"]).groupby(
    ["station", "year", "month"], dropna=True
)
summary = g[value_cols].agg(["mean", "count"]).round(2)
summary.head(12)

In [ ]:
monthly_rain = (
    df.dropna(subset=["station", "year", "month"])
    .groupby(["station", "year", "month"])["Rainfall (mm)"]
    .sum(min_count=1)
    .rename("rainfall_sum_mm")
    .reset_index()
    .sort_values(["station", "year", "month"])
)
monthly_rain.head(10)